In [1]:
import sys, json, datetime
sys.path.append("../src")

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import lightgbm as lgb

from features import RAW_FEATURES
from metrics import evaluate
from threshold import best_threshold, summarize
from reference import build_reference

feat = pd.read_parquet("../data/processed/features.parquet")
tr = feat[feat.split == "train"]
va = feat[feat.split == "val"]

X_tr, y_tr = tr[RAW_FEATURES], tr["isFraud"]
X_va, y_va = va[RAW_FEATURES], va["isFraud"]
amt_va = va["TransactionAmt"].to_numpy()

print(len(RAW_FEATURES), "features |", X_tr.shape, X_va.shape)

52 features | (220806, 52) (91768, 52)


In [2]:
champion = lgb.LGBMClassifier(
    n_estimators=247,            # fixed, from Day 3's early stopping
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
champion.fit(X_tr, y_tr)

val_scores = champion.predict_proba(X_va)[:, 1]
base_metrics = evaluate(y_va, val_scores)
base_metrics

{'pr_auc': 0.6295, 'roc_auc': 0.9181, 'recall@80%prec': 0.467}

In [3]:
REVIEW_COST = 5.0     # assumption: $5 to review one alert

thr, net = best_threshold(y_va, amt_va, val_scores, REVIEW_COST)
summary = summarize(y_va, amt_va, val_scores, thr, REVIEW_COST)
summary

{'threshold': 0.0173,
 'alerts': 17626,
 'alert_rate': 0.1921,
 'precision': 0.1777,
 'recall': 0.8468,
 'fraud_usd_caught': 435174.33,
 'fraud_usd_total': 515459.25,
 'net_benefit': 347044.33}

In [4]:
rows = []
for cost in [1, 2, 5, 10, 25, 50]:
    t, n = best_threshold(y_va, amt_va, val_scores, cost)
    s = summarize(y_va, amt_va, val_scores, t, cost)
    rows.append({"review_cost": cost, **s})

pd.DataFrame(rows)[["review_cost", "threshold", "alert_rate", "precision", "recall", "net_benefit"]]

,review_cost,threshold,alert_rate,precision,recall,net_benefit
0,1,0.0040,0.6051,0.0645,0.9686,449613.60
1,2,0.0082,0.3672,0.1016,0.9257,412544.46
2,5,0.0173,0.1921,0.1777,0.8468,347044.33
3,10,0.0315,0.1147,0.2700,0.7678,282399.07
4,25,0.0761,0.0554,0.4712,0.6473,176813.07
5,50,0.1714,0.0319,0.6844,0.5422,89717.26


In [6]:
flag_all = (amt_va * y_va.to_numpy()).sum() - REVIEW_COST * len(y_va)
print("Flag nothing: net 0 | Flag everything:", round(flag_all, 2))
print("Our threshold:", summary["net_benefit"])

Flag nothing: net 0 | Flag everything: 56619.25
Our threshold: 347044.33


In [7]:
importance = pd.Series(champion.booster_.feature_importance(importance_type="gain"),
                       index=RAW_FEATURES)

ref = build_reference(X_tr, RAW_FEATURES, val_scores, importance)
print(ref["top_features"])

joblib.dump(champion, "../models/champion_v1.joblib")
with open("../models/reference_stats.json", "w") as f:
    json.dump(ref, f)

registry = [{
    "version": "v1",
    "created": str(datetime.date.today()),
    "trained_on_days": [0, 60],
    "n_features": len(RAW_FEATURES),
    "n_estimators": 247,
    "review_cost": REVIEW_COST,
    "threshold": summary["threshold"],
    "val_metrics": base_metrics,
    "val_summary": summary,
    "status": "champion",
}]
with open("../models/registry.json", "w") as f:
    json.dump(registry, f, indent=2)

['C13', 'C1', 'card2', 'addr1', 'card1', 'C14', 'TransactionAmt', 'D3', 'R_emaildomain', 'C11']


In [8]:
m = joblib.load("../models/champion_v1.joblib")
print("Scores match:", np.allclose(m.predict_proba(X_va)[:, 1], val_scores))
print("Features in reference:", len(json.load(open("../models/reference_stats.json"))["features"]))

Scores match: True
Features in reference: 52


In [10]:
import os
for f in ["champion_v1.joblib", "reference_stats.json", "registry.json"]:
    print(f, round(os.path.getsize(f"../models/{f}") / 1e6, 2), "MB")

champion_v1.joblib 1.71 MB
reference_stats.json 0.02 MB
registry.json 0.0 MB


With a $5 review cost (assumption), the cost-optimal threshold on validation is 0.0173, flagging 19.2% of payments with 17.8% precision and 84.7% recall, catching about 84% of fraud dollars. This is much better than flagging nothing (net $0) or everything (net $57K). The rule flags generously because the average fraud amount far exceeds the review cost. Validation net benefit is optimistic because the threshold was tuned on the same rows. Sensitivity: at $25 per review, the threshold rises to 0.076, flagging 5.5% with 47% precision and 65% recall.